#### Determine the classes of a predicted output

In [ ]:
import nibabel as nib
import numpy as np

# Load prediction file
pred = nib.load(r'C:\Users\Waluigi\Desktop\github_repos\TransUNet\data\project_TransUNet\predictions\TU_Synapse224\TU_pretrain_R50-ViT-B_16_skip3_epo150_bs24_224\case0029_pred.nii.gz').get_fdata()

# Get unique classes (excluding background if it's 0)
unique_classes = np.unique(pred)
print(f"Total unique values: {len(unique_classes)}")
print(f"Classes: {unique_classes}")
print(f"Range: {pred.min()} to {pred.max()}")

# Count pixels per class
print("\nPixel count per class:")
for cls in unique_classes:
    count = np.sum(pred == cls)
    print(f"  Class {int(cls)}: {count:,} pixels ({count/pred.size*100:.2f}%)")

# Check if these are valid TransUNet classes (Synapse dataset)
# Expected: 0-8 (9 classes total: 0=background, 1-8=organs)
if pred.max() <= 8:
    print(f"\n✓ Valid TransUNet Synapse prediction (0-8, {int(pred.max())+1} classes)")
else:
    print(f"\n⚠ Unexpected class values (max={pred.max()})")

#### Inspect .npz file format

In [ ]:
import numpy as np
import os
from glob import glob

npz_dir = "./data/Synapse/train_npz"
npz_files = sorted(glob(os.path.join(npz_dir, "*.npz")))

print(f"Found {len(npz_files)} slices")
print("=" * 50)

for npz_file in npz_files:
    with np.load(npz_file, allow_pickle=True) as data:
        filename = os.path.basename(npz_file)
        
        # Get image size
        if 'image' in data:
            img_shape = data['image'].shape
        else:
            img_shape = "No image"
        
        # Get number of unique labels
        if 'label' in data:
            label_array = data['label']
            unique_labels = np.unique(label_array)
            num_labels = len(unique_labels)
        else:
            num_labels = 0
        
        print(f"{filename:30s} Size: {str(img_shape):15s} Labels: {num_labels}")

print("\n" + "=" * 50)
print("SUMMARY:")
print(f"Total slices: {len(npz_files)}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def visualize_slice(npz_path, show_overlay=True):
    """Visualize a single .npz slice"""
    
    with np.load(npz_path) as data:
        image = data['image']
        label = data['label']

        print(image.dtype)
        print(label.dtype)
        
        print(f"📊 File: {npz_path}")
        print(f"Image shape: {image.shape}, range: [{image.min():.3f}, {image.max():.3f}]")
        print(f"Label shape: {label.shape}")
        print(f"Unique labels: {np.unique(label)}")
        
        # Create figure
        n_plots = 3 if show_overlay else 2
        fig, axes = plt.subplots(1, n_plots, figsize=(15, 5))
        
        # Plot 1: CT Image
        im1 = axes[0].imshow(image, cmap='gray')
        axes[0].set_title(f"CT Image\n{image.shape}")
        axes[0].axis('off')
        plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)
        
        # Plot 2: Label Mask
        im2 = axes[1].imshow(label, cmap='viridis', vmin=0, vmax=8)
        axes[1].set_title(f"Labels (9 classes)\nUnique: {np.unique(label)}")
        axes[1].axis('off')
        plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)
        
        # Plot 3: Overlay (optional)
        if show_overlay:
            axes[2].imshow(image, cmap='gray', alpha=0.7)
            # Create colored mask (skip background class 0)
            mask = label > 0
            axes[2].imshow(label, cmap='jet', alpha=0.5, vmin=0, vmax=8)
            axes[2].set_title("Overlay (Image + Labels)")
            axes[2].axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Print label statistics
        print("\n📈 Label Statistics:")
        unique, counts = np.unique(label, return_counts=True)
        for val, count in zip(unique, counts):
            percentage = count / label.size * 100
            print(f"  Class {int(val)}: {count:,} pixels ({percentage:.2f}%)")

# Usage:
visualize_slice("./data/Synapse/train_npz/case0009_slice058.npz")